# Loading the Telstra disruption data into Neo4j

This notebook is the **first step** in the project. It takes the raw Telstra Network Disruptions dataset and turns it into a connected Neo4j graph that the exploration ([analysis.ipynb](analysis.ipynb)) and prediction ([predictor.ipynb](predictor.ipynb)) notebooks build on.

The raw data arrives as flat tables - one row per event-and-attribute pairing - which hides how everything relates. We reshape it into a **property graph**: each disruption event becomes a node, connected to the location it happened at, the alarm it raised, and the log-features, event types and resource types involved. We also build a location-to-location **cascade** that captures how faults tend to follow one another across the network.

The loader is **safe to re-run**: it wipes any previous load, recreates the keys that keep records unique, and streams everything back in batches. Running it top to bottom always produces the same clean graph. Connection details and the data location are read from a local `.env` file.

In [1]:
# Load .env
from dotenv import load_dotenv
import os

load_dotenv()
DATA_DIR = os.getenv("DATA_DIR")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

## Download the raw data

Pull the dataset straight from the Kaggle competition and unpack it into the local data folder. (This step needs valid Kaggle API credentials; once the files are present it can be skipped on re-runs.)

In [2]:
# Download the Telstra dataset

import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

# Initialize and authenticate the API
api = KaggleApi()
api.authenticate()

# Define the competition name
competition_name = "telstra-recruiting-network"

# Download all files associated with the competition
# This downloads a single .zip file to the specified path
api.competition_download_files(competition_name, path=DATA_DIR)

# Unzip the downloaded dataset
zip_path = f"{DATA_DIR}/{competition_name}.zip"
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

# Unzip all files in DATA_DIR
for item in os.listdir(DATA_DIR):
    if item.endswith(".zip"):
        file_path = os.path.join(DATA_DIR, item)
        with zipfile.ZipFile(file_path, "r") as zip_ref:
            zip_ref.extractall(DATA_DIR)

print("Download and extraction complete!")

Download and extraction complete!


## The source tables

The dataset comes as five flat files, all keyed by event `id`:

| file | what it holds | rows per event |
|---|---|---|
| `train` | each event's location and its known fault severity | one |
| `severity_type` | the alarm level raised when the event fired | one |
| `resource_type` | the network resource(s) involved | one or more |
| `log_feature` | the log codes the event produced, each with a volume count | one or more |
| `event_type` | the high-level classification(s) of the event | one or more |

We load each file, then merge them into one wide table purely to sanity-check that the joins line up before building the graph.

In [3]:
# Load all Telstra recruiting network data files
import pandas as pd
import os

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
severity = pd.read_csv(os.path.join(DATA_DIR, "severity_type.csv"))
resource = pd.read_csv(os.path.join(DATA_DIR, "resource_type.csv"))
log = pd.read_csv(os.path.join(DATA_DIR, "log_feature.csv"))
event = pd.read_csv(os.path.join(DATA_DIR, "event_type.csv"))

In [4]:
# Merge all auxiliary tables on 'id' to create unified datasets
# Many-to-one relationships exist for event_type, log_feature, and resource_type
train_df = (
    train.merge(severity, on="id", how="left")
    .merge(resource, on="id", how="left")
    .merge(log, on="id", how="left")
    .merge(event, on="id", how="left")
)

print("Unified Train dataframe shape:", train_df.shape)

Unified Train dataframe shape: (61839, 8)


## The graph model

We reshape those flat tables into a **property graph**. The disruption event is the central entity; everything we know about it hangs off it as a connection, and locations are additionally linked to each other by how faults cascade between them.

**Nodes**

* `:DisruptionEvent` - the core entity (one per event id); the thing we later predict on
* `:Location` - where the event occurred
* `:LogFeature` - the log / error codes generated
* `:EventType` - the high-level classification of what happened
* `:ResourceType` - the network resource affected (e.g. router, switch, fibre link)
* `:SeverityType` - the alarm level of the log message itself

**Relationships**

* `(:DisruptionEvent)-[:OCCURRED_AT]->(:Location)`
* `(:DisruptionEvent)-[:LOGGED {volume}]->(:LogFeature)` - the edge records *how much* of that feature was logged
* `(:DisruptionEvent)-[:TRIGGERED]->(:EventType)`
* `(:DisruptionEvent)-[:ON_RESOURCE]->(:ResourceType)`
* `(:DisruptionEvent)-[:HAS_ALARM]->(:SeverityType)`
* `(:Location)-[:NEXT_EVENT {weight}]->(:Location)` - the **cascade** edge. When an event at site A is immediately followed (in id order) by one at site B, we link A→B; repeated transitions add up in `weight`, giving a proxy for how faults ripple from one part of the network to the next.

## Load the graph into Neo4j

We now translate the tables into the graph described above, in four passes: the events and their locations, the attribute nodes and their connections, and finally the location cascade. The loader is **idempotent** - it first wipes any previous load, (re)creates the uniqueness constraints that keep records unique (these double as indexes, which keep the loads fast), then streams everything in batches so we never send one oversized payload to the database. Re-running always yields the same clean graph.

In [5]:
# Connect to Neo4j via the shared helper package
from neo4j_analysis import Neo4jAnalysis

graph = Neo4jAnalysis(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE)
assert graph.verify_connection(), "Could not connect to Neo4j - check .env credentials"
print("Connected to", NEO4J_URI)

Connected to neo4j+s://d7410db2.databases.neo4j.io


In [6]:
# Reset the graph and (re)create uniqueness constraints.
# Constraints double as indexes, which makes the MERGE-by-key loads below fast.
graph.run_query(
    "MATCH (n) CALL { WITH n DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS"
)

constraints = [
    "CREATE CONSTRAINT event_id     IF NOT EXISTS FOR (e:DisruptionEvent) REQUIRE e.id   IS UNIQUE",
    "CREATE CONSTRAINT location_name IF NOT EXISTS FOR (l:Location)       REQUIRE l.name IS UNIQUE",
    "CREATE CONSTRAINT logfeature_name IF NOT EXISTS FOR (f:LogFeature)   REQUIRE f.name IS UNIQUE",
    "CREATE CONSTRAINT eventtype_name  IF NOT EXISTS FOR (t:EventType)    REQUIRE t.name IS UNIQUE",
    "CREATE CONSTRAINT resourcetype_name IF NOT EXISTS FOR (r:ResourceType) REQUIRE r.name IS UNIQUE",
    "CREATE CONSTRAINT severitytype_name IF NOT EXISTS FOR (s:SeverityType) REQUIRE s.name IS UNIQUE",
]
for c in constraints:
    graph.run_query(c)

print("Graph wiped, constraints in place.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (n) { ... }', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n) CALL { WITH n DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS'


Graph wiped, constraints in place.


In [7]:
# Small helper: stream a list of row-dicts through an UNWIND query in batches so we
# never ship one giant payload over the wire to Aura.
from tqdm.auto import tqdm


def load_in_batches(query, rows, batch_size=2000, desc=""):
    for i in tqdm(range(0, len(rows), batch_size), desc=desc):
        graph.run_query(query, {"rows": rows[i : i + batch_size]})

In [8]:
# 1) DisruptionEvent + Location + (:DisruptionEvent)-[:OCCURRED_AT]->(:Location)
# Train events carry the `fault_severity` target; test events get null so we can tell
# the two populations apart later with a single `dataset` property.
train_events = [
    {"id": int(r.id), "location": r.location,
     "fault_severity": int(r.fault_severity), "dataset": "train"}
    for r in train.itertuples()
]

all_events = train_events

load_in_batches(
    """
    UNWIND $rows AS row
    MERGE (e:DisruptionEvent {id: row.id})
      SET e.dataset = row.dataset,
          e.fault_severity = row.fault_severity
    MERGE (l:Location {name: row.location})
    MERGE (e)-[:OCCURRED_AT]->(l)
    """,
    all_events,
    desc="events + locations",
)
print(f"Loaded {len(all_events):,} disruption events "
      f"({len(train_events):,} train")

events + locations:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded 7,381 disruption events (7,381 train


In [9]:
# 2) Dimension nodes + their relationships to each event.
#    severity_type  -> exactly one per event  : (e)-[:HAS_ALARM]->(:SeverityType)
#    resource_type  -> many per event         : (e)-[:ON_RESOURCE]->(:ResourceType)
#    event_type     -> many per event         : (e)-[:TRIGGERED]->(:EventType)
#    log_feature    -> many per event, w/ vol : (e)-[:LOGGED {volume}]->(:LogFeature)

load_in_batches(
    """
    UNWIND $rows AS row
    MATCH (e:DisruptionEvent {id: row.id})
    MERGE (s:SeverityType {name: row.severity_type})
    MERGE (e)-[:HAS_ALARM]->(s)
    """,
    [{"id": int(r.id), "severity_type": r.severity_type} for r in severity.itertuples()],
    desc="severity",
)

load_in_batches(
    """
    UNWIND $rows AS row
    MATCH (e:DisruptionEvent {id: row.id})
    MERGE (r:ResourceType {name: row.resource_type})
    MERGE (e)-[:ON_RESOURCE]->(r)
    """,
    [{"id": int(r.id), "resource_type": r.resource_type} for r in resource.itertuples()],
    desc="resource",
)

load_in_batches(
    """
    UNWIND $rows AS row
    MATCH (e:DisruptionEvent {id: row.id})
    MERGE (t:EventType {name: row.event_type})
    MERGE (e)-[:TRIGGERED]->(t)
    """,
    [{"id": int(r.id), "event_type": r.event_type} for r in event.itertuples()],
    desc="event_type",
)

load_in_batches(
    """
    UNWIND $rows AS row
    MATCH (e:DisruptionEvent {id: row.id})
    MERGE (f:LogFeature {name: row.log_feature})
    MERGE (e)-[rel:LOGGED]->(f)
      SET rel.volume = row.volume
    """,
    [{"id": int(r.id), "log_feature": r.log_feature, "volume": int(r.volume)}
     for r in log.itertuples()],
    desc="log_feature",
)
print("Dimension relationships loaded.")

severity:   0%|          | 0/10 [00:00<?, ?it/s]

resource:   0%|          | 0/11 [00:00<?, ?it/s]

event_type:   0%|          | 0/16 [00:00<?, ?it/s]

log_feature:   0%|          | 0/30 [00:00<?, ?it/s]

Dimension relationships loaded.


In [10]:
# 3) (:Location)-[:NEXT_EVENT {weight}]->(:Location)
# Order *all* events (train + test) by id, then for each consecutive pair draw a
# directed edge between the locations they occurred at. Repeated transitions
# accumulate weight, giving a proxy for how faults cascade across the network.
# Self-transitions (same location twice in a row) are skipped - they describe a
# location's own churn, not a cascade between sites.
from collections import Counter

ordered = (
    pd.concat([train[["id", "location"]]])
    .sort_values("id")
)
locs = ordered["location"].tolist()

pairs = Counter(
    (a, b) for a, b in zip(locs[:-1], locs[1:]) if a != b
)
next_rows = [{"a": a, "b": b, "w": w} for (a, b), w in pairs.items()]

load_in_batches(
    """
    UNWIND $rows AS row
    MATCH (a:Location {name: row.a})
    MATCH (b:Location {name: row.b})
    MERGE (a)-[rel:NEXT_EVENT]->(b)
      SET rel.weight = row.w
    """,
    next_rows,
    desc="next_event",
)
print(f"Created {len(next_rows):,} distinct NEXT_EVENT transitions "
      f"(from {sum(pairs.values()):,} consecutive cross-location pairs)")

next_event:   0%|          | 0/4 [00:00<?, ?it/s]

Created 7,077 distinct NEXT_EVENT transitions (from 7,356 consecutive cross-location pairs)


## Verify the load

A final sanity check before the other notebooks rely on this graph: counts of nodes by type, relationships by type, and events by split. These should line up with the source data - for example, exactly one `OCCURRED_AT` and one `HAS_ALARM` per event.

In [11]:
# 4) Verify the load: node counts by label and relationship counts by type.
print("Nodes by label")
display(graph.run_query_df(
    "MATCH (n) UNWIND labels(n) AS label "
    "RETURN label, count(*) AS count ORDER BY count DESC"
))

print("\nRelationships by type")
display(graph.run_query_df(
    "MATCH ()-[r]->() RETURN type(r) AS relationship, count(*) AS count ORDER BY count DESC"
))

print("\nEvents by dataset split")
display(graph.run_query_df(
    "MATCH (e:DisruptionEvent) RETURN e.dataset AS dataset, count(*) AS count ORDER BY dataset"
))

graph.close()
print("\nLoad complete - Neo4j connection closed.")

Nodes by label


,label,count
0,DisruptionEvent,7381
1,Location,929
2,LogFeature,331
3,EventType,49
4,ResourceType,10
5,SeverityType,5



Relationships by type


,relationship,count
0,LOGGED,23851
1,TRIGGERED,12468
2,ON_RESOURCE,8460
3,OCCURRED_AT,7381
4,HAS_ALARM,7381
5,NEXT_EVENT,7077



Events by dataset split


,dataset,count
0,train,7381



Load complete - Neo4j connection closed.
